## 2. Calidad de datos y limpieza
Se revisa el conjunto de datos para detectar problemas que puedan afectar el análisis o el desempeño de los modelos, se evalúan/revisan los siguientes aspectos:

- presencia de valores nulos
- registros duplicados
- unicidad del identificador `ID`
- consistencia en variables categóricas como `SEX`, `EDUCATION` y `MARRIAGE`
- plausibilidad de la variable `AGE`
- presencia de valores fuera de rango en variables numéricas
- consistencia en nombres de columnas y estructura temporal de las variables de pago y facturación


In [8]:
import pandas as pd
df = pd.read_excel(r"C:\Users\Emiliano\Desktop\BLUEPRUEBA\default of credit card clients.xls", header=1)

esperados_sex = {1, 2}
esperados_education = {1, 2, 3, 4, 5, 6}
esperados_marriage = {1, 2, 3}

print("Duplicados exactos:", df.duplicated().sum())
print("IDs duplicados:", df["ID"].duplicated().sum())

print("\nSEX - conteo:")
print(df["SEX"].value_counts().sort_index().to_string())
fuera_sex = df[~df["SEX"].isin(esperados_sex)]
print(f"Valores fuera del diccionario: {fuera_sex['SEX'].unique().tolist()} ({len(fuera_sex)} registros)")

print("\nEDUCATION - conteo:")
print(df["EDUCATION"].value_counts().sort_index().to_string())
fuera_edu = df[~df["EDUCATION"].isin(esperados_education)]
print(f"Valores fuera del diccionario: {fuera_edu['EDUCATION'].unique().tolist()} ({len(fuera_edu)} registros)")

print("\nMARRIAGE - conteo:")
print(df["MARRIAGE"].value_counts().sort_index().to_string())
fuera_mar = df[~df["MARRIAGE"].isin(esperados_marriage)]
print(f"Valores fuera del diccionario: {fuera_mar['MARRIAGE'].unique().tolist()} ({len(fuera_mar)} registros)")

print("\nEdad mínima:", df["AGE"].min())
print("Edad máxima:", df["AGE"].max())

print("\nValores únicos en columnas PAY:")
cols_pay = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
for col in cols_pay:
    print(f"{col}: {sorted(df[col].unique().tolist())}")

Duplicados exactos: 0
IDs duplicados: 0

SEX - conteo:
SEX
1    11888
2    18112
Valores fuera del diccionario: [] (0 registros)

EDUCATION - conteo:
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Valores fuera del diccionario: [0] (14 registros)

MARRIAGE - conteo:
MARRIAGE
0       54
1    13659
2    15964
3      323
Valores fuera del diccionario: [0] (54 registros)

Edad mínima: 21
Edad máxima: 79

Valores únicos en columnas PAY:
PAY_0: [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
PAY_2: [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
PAY_3: [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
PAY_4: [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
PAY_5: [-2, -1, 0, 2, 3, 4, 5, 6, 7, 8]
PAY_6: [-2, -1, 0, 2, 3, 4, 5, 6, 7, 8]


In [9]:
cols_numericas = ["LIMIT_BAL", "AGE",
                  "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
                  "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]
display(df[cols_numericas].describe().T)

,count,mean,std,min,25%,50%,75%,max
LIMIT_BAL,30000.0,167484.322667,129747.661567,10000.0,50000.00,140000.0,240000.00,1000000.0
AGE,30000.0,35.485500,9.217904,21.0,28.00,34.0,41.00,79.0
BILL_AMT1,30000.0,51223.330900,73635.860576,-165580.0,3558.75,22381.5,67091.00,964511.0
BILL_AMT2,30000.0,49179.075167,71173.768783,-69777.0,2984.75,21200.0,64006.25,983931.0
BILL_AMT3,30000.0,47013.154800,69349.387427,-157264.0,2666.25,20088.5,60164.75,1664089.0
BILL_AMT4,30000.0,43262.948967,64332.856134,-170000.0,2326.75,19052.0,54506.00,891586.0
BILL_AMT5,30000.0,40311.400967,60797.155770,-81334.0,1763.00,18104.5,50190.50,927171.0
BILL_AMT6,30000.0,38871.760400,59554.107537,-339603.0,1256.00,17071.0,49198.25,961664.0
PAY_AMT1,30000.0,5663.580500,16563.280354,0.0,1000.00,2100.0,5006.00,873552.0
PAY_AMT2,30000.0,5921.163500,23040.870402,0.0,833.00,2009.0,5000.00,1684259.0


In [10]:
# ID: no sirve como feature, se elimina
df.drop(columns=["ID"], inplace=True)

# SEX: valores fuera del diccionario → NaN
df.loc[~df["SEX"].isin(esperados_sex), "SEX"] = None

# EDUCATION: valores fuera del diccionario → NaN
df.loc[~df["EDUCATION"].isin(esperados_education), "EDUCATION"] = None

# MARRIAGE: valores fuera del diccionario → NaN
df.loc[~df["MARRIAGE"].isin(esperados_marriage), "MARRIAGE"] = None

# AGE: fuera de rango lógico → NaN
print("Registros con edad fuera de rango (<20 o >100):", 
      df[(df["AGE"] < 20) | (df["AGE"] > 100)].shape[0])
df.loc[(df["AGE"] < 20) | (df["AGE"] > 100), "AGE"] = None

# ── CONFIRMACIÓN FINAL ────────────────────────────────────────
print("\nNulos tras limpieza:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nShape final:", df.shape)
print("SEX después:", sorted(df["SEX"].dropna().unique().tolist()))
print("EDUCATION después:", sorted(df["EDUCATION"].dropna().unique().tolist()))
print("MARRIAGE después:", sorted(df["MARRIAGE"].dropna().unique().tolist()))

Registros con edad fuera de rango (<20 o >100): 0

Nulos tras limpieza:
EDUCATION    14
MARRIAGE     54
dtype: int64

Shape final: (30000, 24)
SEX después: [1.0, 2.0]
EDUCATION después: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
MARRIAGE después: [1.0, 2.0, 3.0]


### HALLAZGO: Las columnas PAY contienen valores -2 que no están en el diccionario original.
El diccionario indica rango de -1 a 9, pero en los datos aparece -2 y no aparece 9.
Se asume que -2 representa un pago anticipado o una variante de "pagó a tiempo".
Por el momento se conservan los valores tal cual sin modificar.

In [12]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Emiliano\Desktop\BLUEPRUEBA\default_limpio.csv")